In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

import subprocess, sys
if not os.path.isdir("/content/repo"):
    subprocess.run(
        ["git", "clone", "https://github.com/arinkc/llm-finetuning-project.git", "/content/repo"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", "/content/repo", "pull"], check=True)
sys.path.insert(0, "/content/repo")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
print(f"✅ Ready | GPU: {torch.cuda.get_device_name(0)}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Ready | GPU: NVIDIA A100-SXM4-40GB


In [2]:
!pip install -q --upgrade \
    "transformers>=4.45.0,<4.50.0" \
    "peft>=0.13.0,<0.15.0" \
    "bitsandbytes" \
    "accelerate>=1.0.0" \
    "datasets>=3.0.0" \
    "sentencepiece"

print("✅ Libraries ready")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 162.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.7 MB/s eta 0:00:00
✅ Libraries ready


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

HF_TOKEN = os.environ["HF_TOKEN"]
BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
FINETUNED_ADAPTER_ID = "Arinkc/pydoc-llama-r16-full"

# 4-bit quantization for inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=HF_TOKEN,
)
base_model.eval()
print(f"   Memory: {base_model.get_memory_footprint() / 1e9:.2f} GB")

print("Loading fine-tuned model (base + LoRA adapter)...")
finetuned_model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},
        token=HF_TOKEN,
    ),
    FINETUNED_ADAPTER_ID,
    token=HF_TOKEN,
)
finetuned_model.eval()
print(f"   Memory: {finetuned_model.get_memory_footprint() / 1e9:.2f} GB")
print("✅ Both models loaded")

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading base model...


...

   Memory: 5.59 GB
Loading fine-tuned model (base + LoRA adapter)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

   Memory: 5.76 GB
✅ Both models loaded


In [3]:
from datasets import load_dataset

print("Loading test set...")
test_ds = load_dataset("Arinkc/pydoc-llama-codesearchnet-curated", split="test")
print(f"Test examples: {len(test_ds)}")
print(f"\nFirst example messages structure:")
print(f"  System: {test_ds[0]['messages'][0]['content'][:60]}...")
print(f"  User: {test_ds[0]['messages'][1]['content'][:80]}...")
print(f"  Reference: {test_ds[0]['messages'][2]['content']}")

Loading test set...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/20.7M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22473 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1248 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1249 [00:00<?, ? examples/s]

Test examples: 1249

First example messages structure:
  System: You are an expert Python documentation writer. Given a Pytho...
  User: Generate a Google-style docstring for this function:

```python
def required(fie...
  Reference: Decorator that checks if return value is set, if not, raises exception.


In [4]:
SYSTEM_PROMPT = """You are an expert Python documentation writer. Given a Python function, generate a concise, Google-style docstring. Output only the docstring text—no surrounding code, no markdown formatting, no preamble."""

def generate_docstring(model, tokenizer, function_code: str, max_new_tokens: int = 200) -> str:
    """Generate a docstring for a given function using the specified model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Generate a Google-style docstring for this function:\n\n```python\n{function_code}\n```"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=1.0,
        )

    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True,
    ).strip()

    return response


# Quick test to make sure generation works
test_fn = """def add(a: int, b: int) -> int:
    return a + b"""

print("Testing base model generation...")
base_out = generate_docstring(base_model, tokenizer, test_fn)
print(f"Base: {base_out[:200]}")

print("\nTesting fine-tuned model generation...")
ft_out = generate_docstring(finetuned_model, tokenizer, test_fn)
print(f"Fine-tuned: {ft_out[:200]}")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Testing base model generation...
Base: Adds two integers.

Args:
    a (int): The first integer to add.
    b (int): The second integer to add.

Returns:
    int: The sum of a and b.

Testing fine-tuned model generation...
Fine-tuned: Add two numbers together.

    Args:
        a: The first number.
        b: The second number.

    Returns:
        The sum of a and b.


In [5]:
import json
from tqdm.auto import tqdm

# Sample 200 examples from the test set
eval_sample = test_ds.shuffle(seed=42).select(range(200))

results = []

print("Generating outputs for 200 test examples...")
for i, example in enumerate(tqdm(eval_sample)):
    # Extract the function code from the user message
    user_content = example['messages'][1]['content']
    # User message format: "Generate a Google-style docstring for this function:\n\n```python\n{code}\n```"
    # Extract just the code
    code_start = user_content.find("```python\n") + len("```python\n")
    code_end = user_content.rfind("\n```")
    function_code = user_content[code_start:code_end] if code_start > 0 else user_content

    reference_docstring = example['messages'][2]['content']

    # Generate from both models
    base_output = generate_docstring(base_model, tokenizer, function_code)
    ft_output = generate_docstring(finetuned_model, tokenizer, function_code)

    results.append({
        "id": i,
        "function_code": function_code,
        "reference": reference_docstring,
        "base_output": base_output,
        "finetuned_output": ft_output,
    })

print(f"✅ Generated {len(results)} outputs from both models")

# Save results
with open("/content/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved to /content/eval_results.json")

Generating outputs for 200 test examples...


  0%|          | 0/200 [00:00<?, ?it/s]

✅ Generated 200 outputs from both models
Saved to /content/eval_results.json


In [6]:
import re

def analyze_output(text: str) -> dict:
    """Analyze a docstring output for quality signals."""
    if not text or not text.strip():
        return {"empty": True}

    stripped = text.strip()

    return {
        "empty": False,
        "length_chars": len(stripped),
        "word_count": len(stripped.split()),

        # Format compliance
        "starts_with_capital": bool(stripped and stripped[0].isupper()),
        "has_args_section": bool(re.search(r'\n\s*Args:\s*\n', text)),
        "has_returns_section": bool(re.search(r'\n\s*Returns?:\s*\n', text)),
        "has_raises_section": bool(re.search(r'\n\s*Raises:\s*\n', text)),
        "has_google_section": bool(re.search(r'\n\s*(?:Args|Returns?|Raises):\s*\n', text)),

        # Anti-patterns (base model failure modes)
        "has_preamble": bool(re.search(
            r'^(here is|here\'s|this is|i\'ve|sure|certainly|of course|below is)',
            stripped.lower()
        )),
        "has_markdown_fence": bool(re.search(r'```', text)),
        "has_rst_style": bool(re.search(r':param\b|:return\b|:rtype\b|:raises\b', text)),
        "has_javadoc_style": bool(re.search(r'@param\b|@return\b', text)),

        # Clean output check
        "is_clean_output": not bool(re.search(
            r'^(here is|here\'s|this is|i\'ve|sure|certainly|```)',
            stripped.lower()
        )),
    }


# Analyze all results
print("Analyzing outputs...")
for r in results:
    r['base_analysis'] = analyze_output(r['base_output'])
    r['ft_analysis'] = analyze_output(r['finetuned_output'])

# Compute aggregate stats
def aggregate_stats(analyses: list, label: str):
    n = len(analyses)
    print(f"\n{'='*50}")
    print(f"{label} ({n} examples)")
    print(f"{'='*50}")

    metrics = [
        ("Starts with capital letter", "starts_with_capital"),
        ("Has Google-style sections", "has_google_section"),
        ("Has Args: section", "has_args_section"),
        ("Has Returns: section", "has_returns_section"),
        ("Clean output (no preamble/fences)", "is_clean_output"),
        ("Has markdown code fences", "has_markdown_fence"),
        ("Has conversational preamble", "has_preamble"),
        ("Has RST-style (:param:)", "has_rst_style"),
    ]

    stats = {}
    for label_str, key in metrics:
        rate = sum(1 for a in analyses if a.get(key, False)) / n
        stats[key] = rate
        print(f"  {label_str:40s} {rate:6.1%}")

    return stats


base_analyses = [r['base_analysis'] for r in results]
ft_analyses = [r['ft_analysis'] for r in results]

base_stats = aggregate_stats(base_analyses, "BASE MODEL (Llama 3.1 8B Instruct)")
ft_stats = aggregate_stats(ft_analyses, "FINE-TUNED MODEL (pydoc-llama-r16-full)")

print(f"\n{'='*50}")
print("IMPROVEMENT SUMMARY")
print(f"{'='*50}")
key_metrics = ["is_clean_output", "starts_with_capital", "has_google_section"]
for key in key_metrics:
    base_rate = base_stats[key]
    ft_rate = ft_stats[key]
    diff = ft_rate - base_rate
    arrow = "↑" if diff > 0 else "↓"
    print(f"  {key:40s} {base_rate:.1%} → {ft_rate:.1%} ({arrow}{abs(diff):.1%})")

Analyzing outputs...

BASE MODEL (Llama 3.1 8B Instruct) (200 examples)
  Starts with capital letter                80.5%
  Has Google-style sections                 78.5%
  Has Args: section                         71.5%
  Has Returns: section                      63.0%
  Clean output (no preamble/fences)         98.0%
  Has markdown code fences                   2.0%
  Has conversational preamble                0.0%
  Has RST-style (:param:)                    0.0%

FINE-TUNED MODEL (pydoc-llama-r16-full) (200 examples)
  Starts with capital letter               100.0%
  Has Google-style sections                 15.5%
  Has Args: section                         11.0%
  Has Returns: section                      11.0%
  Clean output (no preamble/fences)        100.0%
  Has markdown code fences                   0.0%
  Has conversational preamble                0.0%
  Has RST-style (:param:)                    0.0%

IMPROVEMENT SUMMARY
  is_clean_output                          98.0% → 

In [7]:
print("SIDE-BY-SIDE EXAMPLES")
print("="*80)

# Show 8 random examples
import random
random.seed(42)
sample_indices = random.sample(range(len(results)), 8)

for i, idx in enumerate(sample_indices, 1):
    r = results[idx]
    print(f"\n{'='*80}")
    print(f"EXAMPLE {i}")
    print(f"{'='*80}")
    print(f"FUNCTION:")
    print(r['function_code'][:400])
    print(f"\nBASE MODEL:")
    print(r['base_output'][:300])
    print(f"\nFINE-TUNED MODEL:")
    print(r['finetuned_output'][:300])
    print(f"\nREFERENCE:")
    print(r['reference'][:200])

SIDE-BY-SIDE EXAMPLES

EXAMPLE 1
FUNCTION:
def crack(mpub,priv,pathtopriv):

        mpub = str(mpub)
        priv = b58d(priv)
        if int(priv[18:26],16) >= 2147483648:
            raise Exception("Private key input is hardened.  Cannot crack up a level from a hardened key.")
        pathtopriv = pathtopriv.lower()
        if 'h' in pathtopriv or "'" in pathtopriv:
            raise Exception("Path input indicates a hardened key. Cann

BASE MODEL:
Cracks a hardened private key to a non-hardened private key by incrementing the derivation path level.

Args:
    mpub (str): The master public key.
    priv (str): The hardened private key.
    pathtopriv (str): The derivation path to the hardened private key.

Raises:
    Exception: If the private

FINE-TUNED MODEL:
Cracks a private key up to the next level in the BIP32 hierarchy.

REFERENCE:
Input mpub is master xpub key string.
        Input priv is xprv string.
        Path is the path string (e.g. 'm/3/6/2') from the mpub to the
  

In [8]:
def check_hallucination_signals(text: str, function_code: str) -> dict:
    """Check for signs the model hallucinated vs stayed faithful."""

    # Check if model invented exception types not in the code
    exceptions_in_code = set(re.findall(r'raise\s+(\w+)', function_code))
    exceptions_in_doc = set(re.findall(r'(?:Raises?|raise)\s*:?\s*\n?\s*(\w+Error|\w+Exception)', text))
    hallucinated_exceptions = exceptions_in_doc - exceptions_in_code - {'Exception', 'ValueError', 'TypeError'}

    # Check for params mentioned in doc but not in function signature
    sig_match = re.search(r'def\s+\w+\s*\(([^)]*)\)', function_code)
    sig_params = set()
    if sig_match:
        params_str = sig_match.group(1)
        sig_params = {p.strip().split(':')[0].split('=')[0].strip()
                     for p in params_str.split(',') if p.strip() and p.strip() not in ('self', 'cls', '*args', '**kwargs')}

    doc_params = set(re.findall(r'^\s+(\w+)\s*(?:\([^)]*\))?\s*:', text, re.MULTILINE))
    hallucinated_params = doc_params - sig_params if sig_params else set()

    return {
        "has_hallucinated_exceptions": len(hallucinated_exceptions) > 0,
        "hallucinated_exceptions": list(hallucinated_exceptions),
        "has_hallucinated_params": len(hallucinated_params) > 0,
        "is_one_liner": len([l for l in text.strip().split('\n') if l.strip()]) <= 2,
        "is_verbose": len(text.split()) > 80,
    }


print("Running hallucination analysis...")
for r in results:
    r['base_hallucination'] = check_hallucination_signals(r['base_output'], r['function_code'])
    r['ft_hallucination'] = check_hallucination_signals(r['finetuned_output'], r['function_code'])

n = len(results)
base_halluc_rate = sum(1 for r in results if r['base_hallucination']['has_hallucinated_exceptions']) / n
ft_halluc_rate = sum(1 for r in results if r['ft_hallucination']['has_hallucinated_exceptions']) / n
base_verbose = sum(1 for r in results if r['base_hallucination']['is_verbose']) / n
ft_verbose = sum(1 for r in results if r['ft_hallucination']['is_verbose']) / n
base_oneliner = sum(1 for r in results if r['base_hallucination']['is_one_liner']) / n
ft_oneliner = sum(1 for r in results if r['ft_hallucination']['is_one_liner']) / n

print(f"\n{'='*55}")
print(f"HALLUCINATION & VERBOSITY ANALYSIS (n={n})")
print(f"{'='*55}")
print(f"{'Metric':<40} {'Base':>7} {'FT':>7} {'Delta':>7}")
print(f"{'-'*55}")
print(f"{'Hallucinated exceptions':<40} {base_halluc_rate:>7.1%} {ft_halluc_rate:>7.1%} {ft_halluc_rate-base_halluc_rate:>+7.1%}")
print(f"{'Verbose outputs (>80 words)':<40} {base_verbose:>7.1%} {ft_verbose:>7.1%} {ft_verbose-base_verbose:>+7.1%}")
print(f"{'One-liner outputs':<40} {base_oneliner:>7.1%} {ft_oneliner:>7.1%} {ft_oneliner-base_oneliner:>+7.1%}")
print(f"{'Starts with capital':<40} {base_stats['starts_with_capital']:>7.1%} {ft_stats['starts_with_capital']:>7.1%} {ft_stats['starts_with_capital']-base_stats['starts_with_capital']:>+7.1%}")
print(f"{'Clean output (no preamble/fences)':<40} {base_stats['is_clean_output']:>7.1%} {ft_stats['is_clean_output']:>7.1%} {ft_stats['is_clean_output']-base_stats['is_clean_output']:>+7.1%}")

Running hallucination analysis...

HALLUCINATION & VERBOSITY ANALYSIS (n=200)
Metric                                      Base      FT   Delta
-------------------------------------------------------
Hallucinated exceptions                    11.0%    0.0%  -11.0%
Verbose outputs (>80 words)                19.5%    0.0%  -19.5%
One-liner outputs                          19.0%   79.0%  +60.0%
Starts with capital                        80.5%  100.0%  +19.5%
Clean output (no preamble/fences)          98.0%  100.0%   +2.0%
